# Módulo 4: Aprendizaje Automático (Machine Learning)
## Semana 1 - Clase 2: Preparación de Variables (Feature Engineering, Normalización y Escalado)

Este documento tiene como propósito introducir los procesos de limpieza y transformación de datos, pasos fundamentales antes de la fase de modelado algorítmico.

### **Objetivos de Aprendizaje:**
1. Comprender la premisa "Basura entra, basura sale" (Garbage in, Garbage out) y la importancia de la calidad de los datos.
2. Aplicar técnicas de identificación y tratamiento de valores faltantes (NaN) y valores atípicos (Outliers).
3. Transformar variables categóricas a formatos numéricos matriciales mediante la codificación One-Hot (Dummy Variables).
4. Estandarizar y normalizar variables numéricas continuas para homogeneizar la magnitud de las características de entrada.

---
## **Sección 1: Carga de Datos "Sucios" y Exploración Inicial**

### **El Problema de los Datos Reales**
En escenarios del mundo real, las bases de datos raramente se encuentran estructuradas de forma perfecta. Frecuentemente presentan registros incompletos, errores de digitación, o valores atípicos ilógicos. Si un modelo de Machine Learning es entrenado con datos defectuosos, sus predicciones carecerán de validez teórica y práctica.

A continuación, se generará de manera procedimental un conjunto de datos simulado de 100 clientes, en el cual se inyectarán errores de forma deliberada para ilustrar estos problemas.

In [ ]:
# 1. Importación de librerías esenciales
import pandas as pd
import numpy as np

# 2. Fijación de semilla aleatoria para reproducibilidad
np.random.seed(42)

# 3. Generación base de 100 registros
n_registros = 100
edades = np.random.normal(loc=35, scale=10, size=n_registros)
ingresos = np.random.normal(loc=50000, scale=15000, size=n_registros)
generos = np.random.choice(['Masculino', 'Femenino', 'Otro'], size=n_registros, p=[0.48, 0.48, 0.04])
paises = np.random.choice(['Mexico', 'Colombia', 'Argentina', 'Chile', 'Peru'], size=n_registros)

df_clientes = pd.DataFrame({
    'Edad': edades,
    'Ingresos': ingresos,
    'Genero': generos,
    'Pais': paises
})

# 4. Inyección deliberada de errores (Ruido y datos faltantes)
# Inserción de valores nulos (NaN)
idx_nan_edad = np.random.choice(n_registros, size=8, replace=False)
df_clientes.loc[idx_nan_edad, 'Edad'] = np.nan

idx_nan_ingresos = np.random.choice(n_registros, size=12, replace=False)
df_clientes.loc[idx_nan_ingresos, 'Ingresos'] = np.nan

# Inserción de valores atípicos (Outliers) ilógicos en la Edad
df_clientes.loc[12, 'Edad'] = 136  # Edad excesiva
df_clientes.loc[45, 'Edad'] = -5   # Edad negativa
df_clientes.loc[88, 'Edad'] = 250  # Edad imposible

# Inserción de valores atípicos en Ingresos
df_clientes.loc[33, 'Ingresos'] = 10000000 # Ingreso desproporcionado

### **Visualización del Conjunto de Datos**
Es fundamental realizar una inspección visual de las primeras filas del DataFrame empleando el método `.head()` para identificar irregularidades en la estructura tabular.

In [ ]:
# Visualización estilizada de los primeros 15 registros
df_clientes.head(15).style.highlight_null(color='red')

**Análisis de la Salida:** 
Al observar los datos renderizados (donde las celdas nulas han sido resaltadas en rojo), se hace evidente la presencia de valores `NaN`. Asimismo, en la fila con índice `12` se observa una edad ilógica ($136$ años). Estas anomalías deben ser tratadas rigurosamente, ya que los algoritmos de Machine Learning no poseen la capacidad semántica para distinguir errores de medición humana.

---
## **Sección 2: Limpieza de Datos (Data Cleaning) y Valores Faltantes**

### **Detección de Errores Matemáticos**
La inspección visual es insuficiente para bases de datos extensas. Se requiere el uso de estadística descriptiva básica para auditar las variables numéricas.

In [ ]:
# Conteo total de valores nulos por columna
print(df_clientes.isnull().sum())

In [ ]:
# Resumen estadístico descriptivo para detectar valores atípicos en mínimos y máximos
df_clientes.describe()

### **Tratamiento de Anomalías y Valores Nulos**
Existen múltiples estrategias metodológicas para manejar la falta de información:
1. **Eliminación (Drop):** Remover registros incompletos. Útil únicamente cuando representan una fracción insignificante del total del conjunto de datos.
2. **Imputación Estocástica o Constante:** Reemplazar los valores nulos con estadísticos de tendencia central (como la media o la mediana) para no perder información representativa de otras variables.

A continuación, se procederá a eliminar los registros con edades ilógicas y a imputar los valores faltantes en la columna `Edad` utilizando la **mediana** poblacional (para evitar que valores extremos sesguen la imputación).

In [ ]:
# 1. Filtrado de valores atípicos (Outliers): Conservar únicamente edades lógicas (entre 0 y 100 años)
df_limpio = df_clientes[((df_clientes['Edad'] >= 0) & (df_clientes['Edad'] <= 100)) | (df_clientes['Edad'].isnull())].copy()

# 2. Imputación de valores nulos en la columna 'Edad' utilizando la mediana
mediana_edad = df_limpio['Edad'].median()
df_limpio['Edad'] = df_limpio['Edad'].fillna(mediana_edad)

# Verificación directa del conteo de nulos en la columna 'Edad' resultante
print(df_limpio['Edad'].isnull().sum())

### **Ejercicio 1: Imputación de la variable Ingresos**
El DataFrame `df_limpio` aún presenta valores nulos en la columna `Ingresos` y un valor atípico de $10,000,000.

**Requerimientos del ejercicio:**
1. Filtrar el DataFrame para eliminar aquellos registros cuyo ingreso sea superior a $500,000 (remoción de outlier extremo).
2. Imputar los valores nulos restantes en la columna `Ingresos` utilizando la **media** (`mean`) aritmética de la columna.
3. Imprimir directamente la suma de valores nulos restantes en todo el DataFrame para corroborar que es $0$.

In [ ]:
# TODO: Escribir el código para filtrado e imputación de 'Ingresos'
# 1. Filtrar outlier
# df_limpio = ...

# 2. Calcular media e imputar
# media_ingresos = ...
# df_limpio['Ingresos'] = ...

# 3. Verificación de ausencia absoluta de nulos
# print(df_limpio.isnull().sum().sum())

---
## **Sección 3: Ingeniería de Características (Feature Engineering)**

### **Codificación de Variables Categóricas**
Los algoritmos de Machine Learning operan mediante cálculo algebraico matricial, lo que exige que la totalidad de los datos de entrada posean formato numérico. Las variables de texto (categóricas) como `Genero` o `Pais` deben ser transformadas.

La técnica **One-Hot Encoding** o creación de variables *Dummy* consiste en expandir una variable categórica en múltiples columnas binarias ($0$ o $1$), donde cada columna representa la existencia o ausencia de una categoría específica.

In [ ]:
# Transformación de la variable 'Genero' en variables binarias dummy
dummies_genero = pd.get_dummies(df_limpio['Genero'], prefix='Gen', dtype=int)

# Visualización de la matriz resultante
display(dummies_genero.head())

### **Ejercicio 2: Codificación y Concatenación**

**Requerimientos del ejercicio:**
1. Generar las variables binarias para la columna `Pais` utilizando `pd.get_dummies()`.
2. Concatenar las matrices `dummies_genero` y las nuevas variables de países al DataFrame `df_limpio` utilizando `pd.concat()` estableciendo el eje correcto (`axis=1`).
3. Eliminar las columnas categóricas originales (`Genero` y `Pais`) del DataFrame resultante utilizando el método `.drop()`.
4. Imprimir la estructura final del objeto utilizando el método `.info()`.

In [ ]:
# TODO: Escribir el código para la transformación integral de variables categóricas
# 1. Generar variables para País
# dummies_pais = ...

# 2. Concatenación matricial a lo largo de las columnas (axis=1)
# df_final = pd.concat([df_limpio, dummies_genero, dummies_pais], axis=1)

# 3. Eliminación de las columnas nominales originales
# df_final = df_final.drop(columns=['Genero', 'Pais'])

# 4. Verificación estructural
# df_final.info()

---
## **Sección 4: Transformación y Escalado de Variables**

### **Sensibilidad de la Magnitud (Scaling)**
Múltiples algoritmos de Machine Learning (como Máquinas de Vectores de Soporte o K-Vecinos Más Cercanos) basan su convergencia matemática en el cálculo de distancias euclidianas. Si una variable, como los `Ingresos` (valores nominales que sobrepasan las decenas de miles), se evalúa frente a la `Edad` (valores nominales en decenas), el algoritmo otorgará artificialmente un mayor peso matemático a los ingresos debido puramente a su magnitud nominal.

**Técnicas de Estandarización Principales:**
1.  **Normalización Min-Max:** Ajusta todos los datos dentro del rango continuo entre $[0, 1]$. 
    *   Fórmula: $V_{nuevo} = \frac{V - V_{min}}{V_{max} - V_{min}}$
2.  **Estandarización (Z-Score):** Transforma la distribución matemática para poseer una media poblacional de $\mu = 0$ y una desviación estándar de $\sigma = 1$.
    *   Fórmula: $Z = \frac{V - \mu}{\sigma}$

In [ ]:
from sklearn.preprocessing import StandardScaler

# Inicialización del objeto transformador
scaler = StandardScaler()

# Nota: Para propósitos demostrativos, si el código del Ejercicio 2 no ha sido ejecutado aún,
# se simulará temporalmente una estructura equivalente empleando columnas numéricas.
if 'df_final' not in locals():
    df_final = df_limpio[['Edad', 'Ingresos']].dropna()

# Ajuste y transformación del conjunto de características numéricas
caracteristicas_numericas = ['Edad', 'Ingresos']
matriz_escalada = scaler.fit_transform(df_final[caracteristicas_numericas])

# Sustitución in-place en el DataFrame
df_final[caracteristicas_numericas] = matriz_escalada

In [ ]:
# Verificación matemática (La media de ambas columnas convergerá a ~ 0 y la desviación estándar a ~ 1)
print(df_final['Edad'].mean())
print(df_final['Edad'].std())
print(df_final['Ingresos'].mean())
print(df_final['Ingresos'].std())

### **Ejercicio 3: Aplicación de Normalización Min-Max**

**Requerimientos del ejercicio:**
1. Importar la clase `MinMaxScaler` desde el submódulo `sklearn.preprocessing`.
2. Instanciar el objeto transformador.
3. Aplicar la transformación empleando `.fit_transform()` sobre la característica numérica `Edad` y sobreescribir el resultado en una nueva variable de columna denominada `Edad_MinMax`.
4. Imprimir directamente el valor mínimo (`.min()`) y máximo (`.max()`) de la nueva columna para corroborar la compresión escalar hacia los límites estrictos de $[0, 1]$.

In [ ]:
# TODO: Escribir el código para aplicar Normalización Min-Max
# from sklearn.preprocessing import ...

# min_max_scaler = ...
# df_final['Edad_MinMax'] = min_max_scaler.fit_transform(df_final[['Edad']])

# print(df_final['Edad_MinMax'].min())
# print(df_final['Edad_MinMax'].max())

---